In [ ]:
from pathlib import Path
from matplotlib import pyplot as plt
from vascx.fundus.loader import RetinaLoader
from vascx.utils.analysis import (
    extract_in_parallel,
)
from random import sample
import numpy as np
from scipy.stats import norm
from scipy.optimize import curve_fit
from scipy.optimize import least_squares
from rtnls_enface.utils.plotting import mpl_grid

Relevant papers

- https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0194702&type=printable
- https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=1339427

In [ ]:
ds_path = Path("../../samples/fundus")

In [ ]:
loader = RetinaLoader.from_folder(ds_path)

In [ ]:
r = loader[1]

In [ ]:
r.arteries.plot_segments()

In [ ]:
def gaussian_2d(x, y, C, m, A, mu, sigma):
    return C + m * x[:,None] - A*(np.exp(-0.5 * ((y[None,:] - mu) / sigma) ** 2) )

# Objective function for least_squares with weights
def residuals_2d(params, x, y, D, weights):
    D_model = gaussian_2d(x, y, *params)
    residual = (D - D_model) * weights[:,None]  # Apply spatial weights to the residuals
    return residual.ravel()  # Flatten the residual to a 1D array

def fit_vessel_2d_gaussian(segment):
    D = segment.profile
    x = np.linspace(-D.shape[0] // 2 + 1, D.shape[0] // 2, D.shape[0])
    y = np.linspace(-D.shape[1] // 2 + 1, D.shape[1] // 2, D.shape[1])
    weights = np.ones_like(x)
    initial_guess = [np.max(D), 0, np.max(D) - np.min(D), 0, segment.median_diameter]
    result = least_squares(residuals_2d, initial_guess, args=(x,y,D, weights))
    return result.x

In [ ]:
def dog_2d(x, y, C, m, A1, A2, mu, sigma1, sigma2):
    """Difference of Gaussians model: primary Gaussian minus secondary Gaussian."""
    gauss1 = A1 * np.exp(-0.5 * ((y[None, :] - mu) / sigma1) ** 2)
    gauss2 = A2 * np.exp(-0.5 * ((y[None, :] - mu) / sigma2) ** 2)
    return C + m * x[:, None] - (gauss1 - gauss2)  # DoG: first Gaussian minus second

def residuals_dog_2d(params, x, y, D, weights):
    """Objective function for least squares with weights."""
    D_model = dog_2d(x, y, *params)
    residual = (D - D_model) * weights[:, None]  # Apply spatial weights to residuals
    return residual.ravel()  # Flatten residual to 1D array

def fit_vessel_2d_dog(segment):
    """Fits a Difference of Gaussians model to a vessel segment."""
    D = segment.profile
    x = np.linspace(-D.shape[0] // 2 + 1, D.shape[0] // 2, D.shape[0])
    y = np.linspace(-D.shape[1] // 2 + 1, D.shape[1] // 2, D.shape[1])
    weights = np.ones_like(x)

    # Initial parameter guesses
    C0 = np.max(D)
    m0 = 0
    A1_0 = np.max(D) - np.min(D)
    A2_0 = A1_0 / 2  # Initial amplitude of second Gaussian (smaller)
    mu0 = 0  # Center of the Gaussian
    sigma1_0 = segment.median_diameter  # Primary Gaussian width
    sigma2_0 = sigma1_0 * 0.5  # Secondary Gaussian width (smaller)

    initial_guess = [C0, m0, A1_0, A2_0, mu0, sigma1_0, sigma2_0]
    result = least_squares(residuals_dog_2d, initial_guess, args=(x, y, D, weights))
    
    return result.x  # Returns optimized parameters

In [ ]:
def plot_profiles_1d(segment, fit_fn=fit_vessel_2d_dog, eval_fn=dog_2d):
    optimized_params = fit_fn(segment)
    y = np.linspace(-segment.profile.shape[1] // 2 + 1, segment.profile.shape[1] // 2, segment.profile.shape[1])

    ids = sample(range(len(segment.profile)), 16)
    xs = np.array(ids) - len(segment.profile) // 2 + 1
    slices = segment.profile[ids,:]
    with mpl_grid(slices.shape[0] // 4, 4, figsize=(10,4)) as (fig, axes):
        for i, ax in enumerate(axes):
            Y = slices[i,:]

            x = xs[i][None]
            ax.plot(y, Y)
            # ax.plot(y, gaussian_2d(x, y , *initial_guess)[0])
            ax.plot(y, eval_fn(x, y , *optimized_params)[0])
            # ax.plot(x, [gaussian_2d(x_i, *popt) for x_i in x])
            # ax.axvline(x=mu-sigma, color='r', linestyle='--', linewidth=2)
            # ax.axvline(x=mu+sigma, color='r', linestyle='--', linewidth=2)

In [ ]:
def plot_profile(segment):
    plt.imshow(segment.profile, cmap='gray')
    optimized_params = fit_vessel_2d_gaussian(segment)
    C, m, A, mu, sigma = optimized_params
    print(mu, sigma, segment.profile.shape)
    plt.axvline(x=segment.profile.shape[1]//2+mu-sigma, color='r', linestyle='--', linewidth=2)
    plt.axvline(x=segment.profile.shape[1]//2+mu+sigma, color='r', linestyle='--', linewidth=2)

In [ ]:
segment = r.arteries.segments[18]

In [ ]:
plot_profile(segment)

In [ ]:
plot_profiles_1d(segment)

In [ ]:
plot_profile(segment)